---

# ⚙️ Configuración Inicial del Notebook

## 🧹 Limpieza de Widgets

Se eliminan todos los widgets previamente definidos para evitar conflictos en la ejecución.

---



In [0]:
%python
dbutils.widgets.removeAll()

---
## Parametrización del Storage Account

Se define un widget de tipo texto para capturar dinámicamente el nombre del Azure Data Lake Storage.

---

In [0]:
create widget text storageName default "adlsdatabricks1803test";

---

## Configuración del Catálogo

Se crea el catálogo principal que será utilizado dentro de la arquitectura Medallion (Bronze, Silver y Gold).

---

In [0]:
DROP CATALOG IF EXISTS maintenance_iot CASCADE;

In [0]:
CREATE CATALOG IF NOT EXISTS maintenance_iot;

---

## 🗂️ Creación de Schemas (Capas Medallion)

Se crean los schemas que representan las diferentes capas del Data Lake:

- **Raw** → Datos crudos provenientes directamente de los sensores.
- **Bronze** → Datos con transformaciones iniciales y estandarización básica.
- **Silver** → Datos limpios, validados y estructurados.
- **Gold** → Datos agregados y listos para consumo analítico.
- **Exploratory** → Espacio destinado para análisis y pruebas controladas.

Esta separación permite mantener orden, trazabilidad y buenas prácticas de ingeniería de datos.

---

In [0]:
CREATE SCHEMA IF NOT EXISTS maintenance_iot.raw;
CREATE SCHEMA IF NOT EXISTS maintenance_iot.bronze;
CREATE SCHEMA IF NOT EXISTS maintenance_iot.silver;
CREATE SCHEMA IF NOT EXISTS maintenance_iot.golden;
CREATE SCHEMA IF NOT EXISTS maintenance_iot.exploratory;



---

## 🌐 Configuración de External Location

Se define una *External Location* que conecta Unity Catalog con Azure Data Lake Storage (ADLS Gen2).

Esto permite:

- Control centralizado de acceso al almacenamiento
- Gobernanza de datos
- Seguridad basada en credenciales
- Administración estructurada por capas del Data Lake

La External Location será utilizada principalmente por la capa **Raw** para almacenar los datos ingeridos desde los sensores.

---

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@adlsdatabricks1803test.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credencial)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@adlsdatabricks1803test.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credencial)
COMMENT 'Ubicación externa para tablas bronze'

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@adlsdatabricks1803test.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credencial)
COMMENT 'Ubicación externa para tablas silver'

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@adlsdatabricks1803test.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credencial)
COMMENT 'Ubicación externa para tablas golden'

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-unit-catalog`
URL 'abfss://unit-catalog@adlsdatabricks1803test.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL credencial)
COMMENT 'Ubicación externa para las tablas unit-catalog del Data Lake';

# 🟫 Paso 1 — Crear tabla Bronze 

In [0]:
USE CATALOG maintenance_iot;
USE SCHEMA bronze;

CREATE TABLE IF NOT EXISTS sensor_events (
    event_id STRING NOT NULL,
    rig_id STRING NOT NULL,
    sensor_id STRING NOT NULL,
    sensor_type STRING NOT NULL,
    value DOUBLE,
    unit STRING,
    event_timestamp TIMESTAMP,
    ingestion_timestamp TIMESTAMP,
    ingestion_date DATE,
    source_file STRING
)
USING DELTA
PARTITIONED BY (ingestion_date)
LOCATION 'abfss://bronze@adlsdatabricks1803test.dfs.core.windows.net/sensor_events'
COMMENT 'Tabla Bronze para eventos IoT de sensores';

# 🥈 Paso 2 — Crear tabla Silver

In [0]:
USE SCHEMA silver;

CREATE TABLE IF NOT EXISTS sensor_events_clean (
    event_id STRING,
    rig_id STRING,
    sensor_id STRING,
    sensor_type STRING,
    value DOUBLE,
    unit STRING,
    event_timestamp TIMESTAMP,
    moving_avg DOUBLE,
    std_dev DOUBLE,
    anomaly_flag STRING,
    ingestion_date DATE
)
USING DELTA
PARTITIONED BY (ingestion_date)
LOCATION 'abfss://silver@adlsdatabricks1803test.dfs.core.windows.net/sensor_events_clean'
COMMENT 'Tabla Silver con reglas de negocio y detección de anomalías';

USE SCHEMA silver;

CREATE TABLE IF NOT EXISTS sensor_events_clean (
    event_id STRING,
    rig_id STRING,
    sensor_id STRING,
    sensor_type STRING,
    value DOUBLE,
    unit STRING,
    event_timestamp TIMESTAMP,
    moving_avg DOUBLE,
    std_dev DOUBLE,
    anomaly_flag STRING,
    ingestion_date DATE
)
USING DELTA
PARTITIONED BY (ingestion_date)
LOCATION 'abfss://silver@adlsdatabricks1803test.dfs.core.windows.net/sensor_events_clean';

#🥇 Paso 3 — Crear tabla Golden


In [0]:
USE SCHEMA golden;

CREATE TABLE IF NOT EXISTS rig_daily_summary (
    rig_id STRING,
    sensor_type STRING,
    ingestion_date DATE,
    avg_value DOUBLE,
    max_value DOUBLE,
    min_value DOUBLE,
    anomaly_count BIGINT
)
USING DELTA
PARTITIONED BY (ingestion_date)
LOCATION 'abfss://golden@adlsdatabricks1803test.dfs.core.windows.net/rig_daily_summary'
COMMENT 'Tabla Golden con agregados diarios por rig';

In [0]:
SHOW TABLES IN maintenance_iot.bronze;
SHOW TABLES IN maintenance_iot.silver;
SHOW TABLES IN maintenance_iot.golden;

database,tableName,isTemporary
bronze,sensor_events,false
